# Ibuprofen in the gas phase

This notebook demonstrates a simple use case of the [Open Force Field (OpenFF)](https://openforcefield.org/) toolkit.
OpenFF is an initiative that aims to bring more transparency into the development of force fields (for small molecules), following open science and open source principles.

We will assign force-field parameters to a small molecule, for which the biomolecular force fields used in the previous notebooks (e.g. AMBER ff14SB) contain no parameters.
For this purpose, we use version 2.2.1 of the [Sage force field](https://openforcefield.org/force-fields/force-fields/), which covers a very broad range of small drug-like organic molecules.
Sage serves the same purpose as established small-molecule force fields such as GAFF, OPLS or CGenFF.
Unlike these, Sage does not rely on atom types: parameters are assigned directly with substructure patterns (SMIRKS), which makes the force field more compact and easier to extend and improve.
More details can be found here:

- The SMIRNOFF format used by OpenFF: [10.1021/acs.jctc.8b00640](https://doi.org/10.1021/acs.jctc.8b00640)
- The Sage force field: [10.1021/acs.jctc.3c00039](https://doi.org/10.1021/acs.jctc.3c00039)

This notebook uses ibuprofen as a simple test case.
In principle, any small drug-like molecule could be used instead, and you are encouraged to try your favorite compound from [PubChem](https://pubchem.ncbi.nlm.nih.gov/).

In [ ]:
%matplotlib widget
import os
from sys import stdout

import matplotlib.pyplot as plt
import mdtraj
import nglview
import openff.toolkit
import openmm as mm
import pandas
import requests
from openmm import app, unit

# Custom functions defined in ligands.py in the current directory.
from ligands import convert_sdf_to_pdb

## 1. Download a molecule from PubChem

The 3D structure of the molecule is downloaded from PubChem as an SDF file, unless this file is already present.
Besides the atomic positions, an SDF file also contains all bonds with their bond orders, which are needed to assign the force-field parameters.
PubChem usually provides the neutral form of a molecule, which is a reasonable choice for a simulation in the gas phase.
In aqueous solution, however, acidic or basic groups may be (de)protonated, depending on the pH.
For example, ibuprofen is a carboxylic acid with a pKa between 4 and 5, so it is mostly deprotonated at physiological pH.
In such cases, the appropriate protonation state must be prepared explicitly before assigning force-field parameters.

The SDF file is also converted to a PDB file, which provides the topology (with simple atom names such as `C1`, `C2`, ...) and the initial positions for the OpenMM simulation below.
The function `convert_sdf_to_pdb` is defined in [ligands.py](ligands.py).
It keeps the order of the atoms unchanged, which is essential because the force-field parameters are assigned in the order of the atoms in the SDF file.

In [ ]:
cid = "3672"  # This is the PubChem compound ID (CID) of ibuprofen.
fn_sdf = f"CID_{cid}.sdf"
if not os.path.isfile(fn_sdf):
    url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/{cid}/record/SDF/?record_type=3d&response_type=save"
    response = requests.get(url, timeout=60)
    # Stop with an error if the download failed, e.g. for a wrong CID.
    response.raise_for_status()
    with open(fn_sdf, "w") as f:
        f.write(response.text)
fn_pdb = f"CID_{cid}.pdb"
convert_sdf_to_pdb(fn_sdf, fn_pdb)

## 2. Assign Sage 2.2.1 parameters with the SMIRNOFF engine

The OpenFF toolkit is only used in this section.

The first step is to create an OpenFF `Molecule` object, here by loading the SDF file.
OpenFF `Molecule` objects can be created in [many different ways](https://docs.openforcefield.org/projects/toolkit/en/stable/users/molecule_cookbook.html).

In [ ]:
# Load the molecule from the SDF file.
molecule = openff.toolkit.Molecule.from_file(fn_sdf)
# Quick and dirty visualization of the molecule.
molecule.visualize()

Now, we derive an OpenMM `System` object, using the Sage force field.
This looks simple, yet a lot is going on under the hood to derive force-field parameters for this specific molecule.

Applying the force field to the molecule results in an [Interchange](https://docs.openforcefield.org/projects/interchange/en/stable/) object, which contains all force-field parameters assigned to the molecule.
This object can be exported to the input formats of several MD codes, of which we only use OpenMM here.

The partial charges in Sage 2.2.1 are AM1-BCC charges, which are computed with a semi-empirical quantum chemistry calculation (AM1) followed by bond charge corrections (BCC).
The OpenFF toolkit uses the `sqm` program from [AmberTools](https://ambermd.org/AmberTools.php) for this purpose, which may take a few seconds for a drug-like molecule.
(Sage 2.3.0 and newer replace this calculation by [NAGL](https://docs.openforcefield.org/projects/nagl/en/stable/), a graph neural network trained to reproduce AM1-BCC charges, which is faster but requires additional dependencies.)

In [ ]:
force_field = openff.toolkit.ForceField("openff-2.2.1.offxml")
interchange = force_field.create_interchange(molecule.to_topology())
system = interchange.to_openmm_system()

## 3. Perform a short molecular dynamics simulation

The code below closely follows the gas-phase simulation of alanine dipeptide in [../02_alanine_dipeptide/01_force_fields.ipynb](../02_alanine_dipeptide/01_force_fields.ipynb), with a few differences:

- The `System` object was created with the OpenFF toolkit instead of the `ForceField` class of OpenMM.
  The topology and the initial positions are still taken from a PDB file.
- No `constraints` argument is needed: Sage constrains the lengths of all bonds involving hydrogen atoms by default, which allows us to take time steps of 2 femtoseconds.
- The simulation is ten times shorter: 10000 steps, i.e. 20 picoseconds.

In [ ]:
# Set up the MD simulation.
pdb = app.PDBFile(fn_pdb)
integrator = mm.LangevinMiddleIntegrator(
    300 * unit.kelvin, 1 / unit.picosecond, 2 * unit.femtoseconds
)
simulation = app.Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)
simulation.minimizeEnergy()

# Write the minimized structure to a PDB file.
# It is used as topology for MDTraj below and can be useful for debugging.
positions = simulation.context.getState(getPositions=True).getPositions()
with open("init_01.pdb", "w") as f:
    app.PDBFile.writeFile(simulation.topology, positions, f)

# Set the reporters collecting the MD output.
simulation.reporters = []
simulation.reporters.append(app.DCDReporter("traj_01.dcd", 100))
simulation.reporters.append(
    app.StateDataReporter(stdout, 1000, step=True, temperature=True, elapsedTime=True)
)
simulation.reporters.append(
    app.StateDataReporter(
        "scalars_01.csv",
        100,
        time=True,
        potentialEnergy=True,
        totalEnergy=True,
        temperature=True,
    )
)
simulation.step(10000)

In the visualization below, the trajectory is first aligned to the first frame, to remove the overall rotation of the molecule.

In [ ]:
traj = mdtraj.load("traj_01.dcd", top="init_01.pdb")
traj.superpose(traj, 0)
view = nglview.show_mdtraj(traj)
view.clear_representations()
view.add_licorice()
view

The temperature as a function of time gives a first impression of the equilibration.
After a short warm-up phase, the temperature fluctuates around 300 K.
These fluctuations are large because the molecule has only a few degrees of freedom.
The relative fluctuations of the instantaneous temperature become smaller as the number of atoms increases.

In [ ]:
df = pandas.read_csv("scalars_01.csv")
df = df.rename(columns={'#"Time (ps)"': "Time (ps)"})
plt.close("temp")
fig, ax = plt.subplots(num="temp")
df.plot(kind="line", x="Time (ps)", y="Temperature (K)", ax=ax);